## Plotly

1. Подключение к db.

In [1]:
import pandas as pd
import sqlite3
import plotly.graph_objects as go
import numpy as np

conn = sqlite3.connect("../data/checking-logs.sqlite")

2. Формирование датафрейма.

In [2]:
query = """
SELECT uid, timestamp, numTrials
FROM checker
WHERE uid NOT LIKE 'admin%' 
AND labname = 'project1'
AND status = 'ready'
"""

df = pd.read_sql(query, conn)

df['timestamp'] = pd.to_datetime(df['timestamp'])
df['date'] = df['timestamp'].dt.date

df_grouped = df.groupby(['uid', 'date']).size().reset_index(name='commits')
df_grouped['cumulative_commits'] = df_grouped.groupby('uid')['commits'].cumsum()

df_grouped.head(10)

,uid,date,commits,cumulative_commits
0,user_1,2020-05-14,11,11
1,user_10,2020-05-12,7,7
2,user_10,2020-05-13,14,21
3,user_10,2020-05-14,37,58
4,user_11,2020-05-03,1,1
5,user_12,2020-05-14,4,4
6,user_13,2020-05-11,2,2
7,user_13,2020-05-12,28,30
8,user_13,2020-05-14,2,32
9,user_14,2020-05-05,1,1


3. Построение графика.

In [3]:
fig = go.Figure()

unique_dates = sorted(df_grouped['date'].unique())
users = df_grouped['uid'].unique()

for user in users:
    user_data = df_grouped[df_grouped['uid'] == user]
    fig.add_trace(go.Scatter(
        x=user_data['date'],
        y=user_data['cumulative_commits'],
        mode='lines',
        name=user,
        line=dict(width=2)
    ))

frames = []
for date in unique_dates:
    frame_data = []
    for user in users:
        user_data = df_grouped[(df_grouped['uid'] == user) & (df_grouped['date'] <= date)]
        frame_data.append(go.Scatter(
            x=user_data["date"],
            y=user_data["cumulative_commits"],
            mode='lines+markers',
            name=user,
            line=dict(width=2),
            marker=dict(size=6)
        ))

    frames.append(go.Frame(
        data=frame_data,
        name=str(date)
    ))

fig.frames = frames

fig.update_layout(
    updatemenus=[{
        "buttons": [
            {
                "args": [None, {"frame": {"duration": 500, "redraw": True}, "fromcurrent": True}],
                "label": "Play",
                "method": "animate"
            },
            {
                "args": [[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}],
                "label": "Pause", 
                "method": "animate"
            }
        ],
        "type": "buttons",
        "x": 0.1,
        "y": 0,
    }]
)

fig.update_layout(
    title="Dynamic of commits per user in project1",
    plot_bgcolor='light gray',
    paper_bgcolor='white',
    font=dict(color='black', size=12),
    xaxis=dict(
        showgrid=True,
        gridcolor='white',
        showline=True,
        linecolor='black'
    ),
    yaxis=dict(
        showgrid=True, 
        gridcolor='white',
        showline=True,
        linecolor='black'
    ),
    showlegend=True,
    height=500,
    width=800
)


fig.show()

In [4]:
conn.close()